In [1]:
%pip install nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import nltk
import pandas as pd
import re
from nltk.corpus import reuters
from nltk.tokenize import sent_tokenize

# -------------------------------
# 1. Download datasets
# -------------------------------
nltk.download('reuters')
nltk.download('punkt')

# -------------------------------
# 2. Define event keywords
# -------------------------------
event_keywords = {
    "earnings": ["profit", "revenue", "sales", "income", "earnings"],
    "merger": ["acquire", "acquisition", "merger", "takeover", "buyout"],
    "rate_change": ["interest rate", "inflation", "fed", "central bank", "rates"]
}

# -------------------------------
# 3. Function to clean text
# -------------------------------
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)  # remove special chars
    text = re.sub(r"\s+", " ", text)            # remove extra spaces
    return text.strip()

# -------------------------------
# 4. Function to assign event label
# -------------------------------
def get_event(sentence):
    for event, keywords in event_keywords.items():
        for word in keywords:
            if word in sentence:
                return event
    return "general"

# -------------------------------
# 5. Extract sentences from Reuters
# -------------------------------
data = []

for fileid in reuters.fileids():
    raw_text = reuters.raw(fileid)
    sentences = sent_tokenize(raw_text)

    for sent in sentences:
        cleaned = clean_text(sent)

        # Filter short sentences
        if len(cleaned.split()) < 5:
            continue

        event = get_event(cleaned)

        data.append([cleaned, event])

# -------------------------------
# 6. Create DataFrame
# -------------------------------
df = pd.DataFrame(data, columns=["Sentence", "Event"])

# -------------------------------
# 7. Remove duplicates
# -------------------------------
df = df.drop_duplicates(subset=["Sentence"])

# -------------------------------
# 8. Balance dataset (optional but recommended)
# # -------------------------------
# min_count = df["Event"].value_counts().min()

# df_balanced = df.groupby("Event").apply(lambda x: x.sample(min_count)).reset_index(drop=True)
min_count = df["Event"].value_counts().min()

df_balanced = df.groupby("Event", group_keys=False).sample(n=min_count).reset_index(drop=True)
# -------------------------------
# 9. Shuffle dataset
# -------------------------------
df_balanced = df_balanced.sample(frac=1).reset_index(drop=True)

# -------------------------------
# 10. Save final dataset
# -------------------------------
df_balanced.to_csv("reuters_events_clean.csv", index=False)

# -------------------------------
# 11. Print summary
# -------------------------------
print("Dataset shape:", df_balanced.shape)
print("\nClass distribution:\n", df_balanced["Event"].value_counts())
print("\n Reuters dataset ready: reuters_events_clean.csv")

[nltk_data] Downloading package reuters to C:\Users\Dileep
[nltk_data]     Samaji\AppData\Roaming\nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Dileep
[nltk_data]     Samaji\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Dataset shape: (11148, 2)

Class distribution:
 Event
rate_change    2787
merger         2787
earnings       2787
general        2787
Name: count, dtype: int64

✅ Reuters dataset ready: reuters_events_clean.csv


In [4]:
print(df.columns)

Index(['Sentence', 'Event'], dtype='str')
